# Notebook 04 — Métricas de fragmentación del paisaje sobre AOI acotado (versión vigente)

Computa las métricas de ecología del paisaje sobre los parches de manglar segmentados por SamGeo en `data/processed/samgeo_acotado/`, utilizando los GeoJSON reproyectados a EPSG:9377 (metros). El cálculo se hace en Julia con la fórmula del shoelace sobre el anillo exterior de cada polígono.

Esta es la **versión limpia** de la fase 4. El notebook `04_fragmentation.ipynb` original contiene también el bloque legacy con aproximación esférica sobre EPSG:4326 que se conserva para trazabilidad del baseline.

**Métricas:** número de parches, área total / media / máxima, densidad por 1.000 ha, MSI medio (índice de forma), NND medio (distancia al vecino más cercano).

**Insumos:** `data/processed/samgeo_acotado/manglar_*_9377.geojson`
**Salida:** `outputs/tables/metricas_fragmentacion_acotado.csv`

In [ ]:
using GeoJSON, DataFrames, Statistics, CSV

base_acotado = "../data/processed/samgeo_acotado"
periodos = ["degradacion", "recuperacion", "actual"]
println("Insumos: $base_acotado")

## Funciones auxiliares

In [ ]:
function shoelace_metros(coords)
    n = length(coords); area = 0.0; perim = 0.0
    for i in 1:n-1
        x1, y1 = coords[i][1], coords[i][2]
        x2, y2 = coords[i+1][1], coords[i+1][2]
        area  += x1 * y2 - x2 * y1
        perim += sqrt((x2 - x1)^2 + (y2 - y1)^2)
    end
    return abs(area) / 2.0, perim
end

function metricas_de(periodo, base)
    path = joinpath(base, "manglar_$(periodo)_9377.geojson")
    isfile(path) || return nothing
    fc = GeoJSON.read(read(path, String))
    areas_ha, perim_m = Float64[], Float64[]
    cx, cy = Float64[], Float64[]
    for feat in fc.features
        geom = feat.geometry
        (geom isa GeoJSON.Polygon && length(geom.coordinates) > 0) || continue
        ring = geom.coordinates[1]
        length(ring) >= 4 || continue
        a_m2, per_m = shoelace_metros(ring)
        a_ha = a_m2 / 10_000.0
        if 1.0 <= a_ha < 5000.0
            push!(areas_ha, a_ha); push!(perim_m, per_m)
            push!(cx, mean([r[1] for r in ring[1:end-1]]))
            push!(cy, mean([r[2] for r in ring[1:end-1]]))
        end
    end
    return areas_ha, perim_m, cx, cy
end

## Cálculo de las métricas por período

In [ ]:
resultados = DataFrame(
    Periodo=String[], Parches=Int[],
    Area_total_ha=Float64[], Area_media_ha=Float64[], Area_max_ha=Float64[],
    Densidad_parches_1000ha=Float64[], MSI_medio=Float64[], NND_medio_km=Float64[])

for p in periodos
    res = metricas_de(p, base_acotado)
    res === nothing && (println("  no encontrado: $p"); continue)
    areas_ha, perim_m, cx, cy = res
    n = length(areas_ha)
    n == 0 && continue
    msi = mean([perim_m[i] / sqrt(pi * areas_ha[i] * 10_000) for i in 1:n])
    # NND - distancia minima entre centroides (km)
    nnd_vals = Float64[]
    for i in 1:n
        min_d = Inf
        for j in 1:n
            i == j && continue
            d = sqrt((cx[i]-cx[j])^2 + (cy[i]-cy[j])^2)
            d < min_d && (min_d = d)
        end
        push!(nnd_vals, min_d / 1000.0)
    end
    densidad = n / (sum(areas_ha) / 1000)
    push!(resultados, (p, n, sum(areas_ha), mean(areas_ha),
                       maximum(areas_ha), densidad, msi, mean(nnd_vals)))
    println("  $p → $n parches, $(round(sum(areas_ha), digits=1)) ha")
end

println("\n=== FRAGMENTACIÓN SOBRE AOI ACOTADO (EPSG:9377) ===")
println(resultados)
CSV.write("../outputs/tables/metricas_fragmentacion_acotado.csv", resultados)
println("\nGuardado: outputs/tables/metricas_fragmentacion_acotado.csv")